In [19]:
!pip install clustering-benchmarks -q

In [20]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [21]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'uci': ['ecoli', 'glass', 'ionosphere', 'sonar', 'statlog', 'wdbc', 'wine', 'yeast'],
                         #'mnist': ['digits', 'fashion'], # Genie digits ~ 32 min
                         #'sipu': ['worms_64'], # Genie ~ 9 min
                         }

## Create autoencoder using pytorch

In [22]:
import torch.nn.functional as F

def vae_loss(x_hat, x, mu, logvar):
    # Reconstruction loss (MSE)
    reconstruction_loss = F.mse_loss(x_hat, x, reduction='sum')

    # KL divergence loss
    # 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_divergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return reconstruction_loss + kl_divergence

def autoencoder_feat_eng(X, n_embeddings=8):

  X = StandardScaler().fit_transform(X)
  X = torch.tensor(X, dtype=torch.float)

  # --- Autoencoder definition ---
  class Autoencoder(nn.Module):
      def __init__(self, input_dim, latent_dim=n_embeddings):
          super().__init__()
          self.encoder = nn.Sequential(
              nn.Linear(input_dim, 64),
              nn.ReLU(),
          )
          self.fc_mu = nn.Linear(64, latent_dim)
          self.fc_logvar = nn.Linear(64, latent_dim)

          self.decoder = nn.Sequential(
              nn.Linear(latent_dim, 64),
              nn.ReLU(),
              nn.Linear(64, input_dim)
          )

      def reparameterize(self, mu, logvar):
          std = torch.exp(0.5 * logvar)
          eps = torch.randn_like(std)
          return mu + eps * std

      def forward(self, x):
          encoded = self.encoder(x)
          mu = self.fc_mu(encoded)
          logvar = self.fc_logvar(encoded)
          z = self.reparameterize(mu, logvar)
          x_hat = self.decoder(z)
          return x_hat, mu, logvar

  # --- Train AE ---
  input_dim = X.shape[1] # Dynamically set input_dim
  model = Autoencoder(input_dim)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


  for epoch in range(100):
      x_hat, mu, logvar = model(X)
      loss = vae_loss(x_hat, X, mu, logvar)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # --- Cluster latent space ---
  with torch.no_grad():
      # We will use the mean as the latent representation for clustering
      latent = model.fc_mu(model.encoder(X)).numpy()

  return latent

### Difference between original dataset and embeddings from the autoencoder

In [23]:
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"
battery = "uci"
dataset = "statlog"
b = clustbench.load_dataset(battery, dataset, url=data_url)

In [24]:
import pandas as pd
pd.DataFrame(b.data)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,1.765321,1.035127,0.001836,-0.000090,-0.020114,-0.097886,-0.024912,-0.146013,0.428179,0.372140,0.588548,0.323849,-0.168113,0.481105,-0.312993,0.570539,-0.002046,-0.012852
1,-0.225939,0.124836,-0.000271,-0.000090,-0.030649,-0.103515,-0.039662,-0.149413,-0.685802,-0.622436,-0.789531,-0.645443,0.190102,-0.311185,0.121082,-0.807540,0.010869,-0.014420
2,1.461891,-1.562993,-0.000272,-0.000089,-0.018007,-0.093630,-0.024911,-0.136886,1.630661,1.499467,1.812803,1.579712,-0.393579,0.546429,-0.152850,1.794795,-0.004314,-0.017769
3,-1.762054,0.940306,-0.000271,-0.000088,-0.003257,-0.074487,0.124696,-0.028337,0.124047,0.127711,0.165010,0.079419,0.010994,0.122890,-0.133885,0.147000,-0.003034,-0.012061
4,-1.212087,1.395451,-0.000272,-0.000089,-0.008524,-0.079535,0.003534,-0.119821,0.237832,0.216212,0.329367,0.167918,-0.064864,0.274605,-0.209743,0.311359,-0.002351,-0.012504
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,-1.799982,-0.406166,-0.000272,-0.000090,-0.012740,-0.106026,-0.020698,-0.141165,-0.318456,-0.236825,-0.363886,-0.354656,0.244888,-0.136290,-0.108598,-0.381895,-0.000869,-0.003645
2306,0.342993,-1.885389,-0.000273,-0.000090,-0.011685,-0.091066,-0.029127,-0.134703,1.717756,1.609041,1.848625,1.695604,-0.326151,0.392606,-0.066455,1.830616,-0.004883,-0.018706
2307,-0.851764,-0.975098,-0.000271,-0.000090,-0.012738,-0.089239,-0.018590,-0.134196,0.416238,0.351068,0.573798,0.323848,-0.195507,0.472678,-0.277170,0.555788,-0.002129,-0.013793
2308,-0.510405,0.181730,-0.000272,-0.000089,-0.025382,-0.105009,-0.038608,-0.150121,-0.684398,-0.622437,-0.785317,-0.645443,0.185889,-0.302756,0.116865,-0.803325,0.010868,-0.014421


In [25]:
pd.DataFrame(autoencoder_feat_eng(b.data))

,0,1,2,3,4,5,6,7
0,-0.022050,-0.527918,0.330906,-0.144954,-0.540290,0.725451,0.389416,-0.216032
1,-0.099095,0.413280,-0.781822,-0.872837,0.581970,-0.749466,-1.069549,0.347503
2,-0.841214,-0.733536,0.678844,-0.177841,-1.347610,2.275497,0.744471,-0.116182
3,0.169365,-0.287665,0.178021,0.232652,-0.154692,-0.063898,0.178544,0.017422
4,0.115736,-0.501395,0.192989,-0.019087,-0.303005,0.430627,0.082342,0.159686
...,...,...,...,...,...,...,...,...
2305,-0.105195,-0.088152,-0.218521,-0.743874,0.225101,-0.271064,-0.554739,0.574963
2306,-0.830679,-0.761470,0.724712,-0.151433,-1.282445,2.237554,0.736058,-0.019854
2307,-0.290123,-0.656652,0.448637,-0.252211,-0.546983,0.932927,0.010380,0.158876
2308,-0.095237,0.412637,-0.759531,-0.847575,0.606837,-0.770001,-1.042083,0.396047


## Function get_scores

Get the NCA score of a specific dataset using genie mst algorithm

In [26]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Function get_scores_with_autoencoder

Get the NCA score of a specific dataset applying the autoencoder transformation to the data and then using genie mst algorithm

In [27]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_autoencoder(battery, dataset, apply_scale=False, n_embeddings=2):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = autoencoder_feat_eng(b.data, n_embeddings=n_embeddings)

  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Execute get_scores on all datasets as baseline

In [28]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Datasets"):
  for dataset in battery_datasets_dict[battery]:
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset))

df = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 2/2 [00:05<00:00,  2.59s/it]


In [29]:
df

,Battery,Dataset,Genie NCA Score
0,fcps,atom,1.000000
1,fcps,chainlink,1.000000
2,fcps,engytime,0.918870
3,fcps,hepta,1.000000
4,fcps,lsun,1.000000
5,fcps,target,1.000000
6,fcps,tetra,1.000000
7,fcps,twodiamonds,0.987500
8,fcps,wingnut,1.000000
9,uci,ecoli,0.435664


## Execute get_scores_with_autoencoder on all datasets trying different numbers of embeddings (last hidden layer of the neural network)



In [30]:
n_embeddings_list = [4,16,32,64,128,256,512]

scores_lists_vae = {}
for n_embs in tqdm.tqdm(n_embeddings_list, desc="Processing Datasets with VAE"):
  for battery in battery_datasets_dict.keys():
    for dataset in battery_datasets_dict[battery]:
      column_name = 'Genie+VAE '+ str(n_embs) +' NCA Score'
      if column_name not in scores_lists_vae:
        scores_lists_vae[column_name] = []
      scores_lists_vae[column_name].append(get_scores_with_autoencoder(battery, dataset, n_embeddings=n_embs))

df_vae = pd.DataFrame.from_dict(scores_lists_vae)

Processing Datasets with VAE: 100%|██████████| 7/7 [02:37<00:00, 22.54s/it]


# Final Results

In [31]:
df = pd.concat([df, df_vae], axis=1)
numerical_cols = df.columns[2:]
df.style.highlight_max(axis=1, subset=numerical_cols)

,Battery,Dataset,Genie NCA Score,Genie+VAE 4 NCA Score,Genie+VAE 16 NCA Score,Genie+VAE 32 NCA Score,Genie+VAE 64 NCA Score,Genie+VAE 128 NCA Score,Genie+VAE 256 NCA Score,Genie+VAE 512 NCA Score
0,fcps,atom,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,fcps,chainlink,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,fcps,engytime,0.918870,0.917404,0.962393,0.917892,0.920822,0.914480,0.914480,0.918390
3,fcps,hepta,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,fcps,tetra,1.000000,0.903333,1.000000,1.000000,1.000000,1.000000,1.000000,0.996667
7,fcps,twodiamonds,0.987500,0.990000,0.990000,0.990000,0.990000,0.990000,0.990000,0.990000
8,fcps,wingnut,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,uci,ecoli,0.435664,0.224154,0.384002,0.327530,0.340502,0.323576,0.319837,0.358805
